In [2]:
import pandas as pd
import numpy as np
import scanpy as sc
import scanpy.external as sce
from matplotlib.pyplot import rc_context
import matplotlib.pyplot as plt
import glob as glob
import scikit_posthocs as sp

In [3]:
from statsmodels.stats import multitest

In [4]:
def initializeDeviations(adata, deviationfile):
    devations = pd.read_csv(deviationfile, index_col=0)

    tfdict = dict()
    jaspardict = dict()
    for curcol in devations.columns:
        split = curcol.split(".")
        
        tfdict[split[-1]] = curcol
        jaspardict[".".join(split[:-1])] = curcol    
        adata.obs[curcol] = np.nan
        
    adata.obs = adata.obs.copy()
    return tfdict, jaspardict

In [5]:
def addDeviations(adata, deviationfile):
    deviations = pd.read_csv(deviationfile, index_col=0)
    
    newindex=[]
    for cur in deviations.index:
        newindex.append(cur.replace(".","-"))

    deviations.index = newindex
    deviations = deviations.sort_index()
    
    for curcol in deviations.columns:
        adata.obs.loc[deviations.index, curcol] = deviations[curcol].values



In [6]:
cd4 = sc.read_h5ad("../Pass2_Annotation/Lifespan_PCA_CD4T.h5ad")
cd8 = sc.read_h5ad("../Pass2_Annotation/Lifespan_PCA_CD8T.h5ad")


In [7]:
cd4_init = initializeDeviations(cd4, "./deviations/CD4_naive_deviations.txt")
cd8_init = initializeDeviations(cd8, "./deviations/CD8_naive_deviations.txt")


/var/folders/6z/4f2kjdcd2pj9cxfbcgy_73vmvwjr__/T/ipykernel_27317/3354187533.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  adata.obs[curcol] = np.nan
/var/folders/6z/4f2kjdcd2pj9cxfbcgy_73vmvwjr__/T/ipykernel_27317/3354187533.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  adata.obs[curcol] = np.nan
/var/folders/6z/4f2kjdcd2pj9cxfbcgy_73vmvwjr__/T/ipykernel_27317/3354187533.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perfor

/var/folders/6z/4f2kjdcd2pj9cxfbcgy_73vmvwjr__/T/ipykernel_27317/3354187533.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  adata.obs[curcol] = np.nan
/var/folders/6z/4f2kjdcd2pj9cxfbcgy_73vmvwjr__/T/ipykernel_27317/3354187533.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  adata.obs[curcol] = np.nan
/var/folders/6z/4f2kjdcd2pj9cxfbcgy_73vmvwjr__/T/ipykernel_27317/3354187533.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perfor

In [7]:
addDeviations(cd4, "./deviations/CD4_naive_deviations.txt")


In [8]:
addDeviations(cd8, "./deviations/CD8_naive_deviations.txt")

In [9]:
#cd4.write("Lifespan_CD4T_chromVAR.h5ad")
#cd8.write("Lifespan_CD8T_chromVAR.h5ad")

In [10]:
cd4 = sc.read_h5ad("Lifespan_CD4T_chromVAR.h5ad")
cd8 = sc.read_h5ad("Lifespan_CD8T_chromVAR.h5ad")


In [11]:
import scipy.stats as stats

In [12]:
def getMedianDeviationsPaired(obj, tf, a1, a2, annotationvar='Annotation'):
    rv1 = []
    rv2 = []
    
    for cursample in np.unique(obj.obs['batch']):
        med_a1 = np.median(obj.obs[tf][(obj.obs['batch'] == cursample) & (obj.obs[annotationvar] == a1)])
        med_a2 = np.median(obj.obs[tf][(obj.obs['batch'] == cursample) & (obj.obs[annotationvar] == a2)])
        if not np.isnan(med_a1) and not np.isnan(med_a2):
            rv1.append(med_a1)
            rv2.append(med_a2)
    return rv1, rv2

In [13]:
def convertMotif(motifstr):
    cursplit = motifstr.split(".")
    motifname = cursplit[-1]
    rv = cursplit[0]
    for i in range(1,len(cursplit)-1):
        rv += "."+cursplit[i]
    rv += "_"+motifname
    return rv

In [14]:
def getPairedDeviationsSummary(adata, a1, a2, tfdict, expressedtfs, annotationvar='Annotation'):
    significantdeviations =  dict()
    significantdeviations['TF'] = []
    significantdeviations['JASPAR ID'] = []

    significantdeviations['Avg. '+a1] = []
    significantdeviations['Avg. '+a2] = []
    significantdeviations['Diff'] = []

    significantdeviations['Wilcoxon'] = []


    for curtf in expressedtfs:
        groups_medians = getMedianDeviationsPaired(adata, tfdict[curtf], a1, a2, annotationvar=annotationvar)

        significantdeviations['TF'].append(curtf)
        significantdeviations['JASPAR ID'].append(".".join(tfdict[curtf].split(".")[:-1]))

        significantdeviations['Avg. '+a1].append(np.mean(groups_medians[0]))
        significantdeviations['Avg. '+a2].append(np.mean(groups_medians[1]))
        significantdeviations['Diff'].append(np.mean(groups_medians[1])-np.mean(groups_medians[0]))

        significantdeviations['Wilcoxon'].append(stats.wilcoxon(groups_medians[0], groups_medians[1])[1])

    significantdeviations_df = pd.DataFrame(significantdeviations)
    return significantdeviations_df

In [8]:
cd4tfdict = cd4_init[0]
cd8tfdict = cd8_init[0]

In [16]:
rna_cd4_adata = sc.read("../Pass2_Annotation/Lifespan_CD4T_scRNA.h5ad")
rna_cd8_adata = sc.read("../Pass2_Annotation/Lifespan_CD8T_scRNA.h5ad")

In [17]:
rna_cd4_adata = rna_cd4_adata[rna_cd4_adata.obs['LS_L4'].isin(['CD4_naive_SOX4-','CD4_naive_SOX4+'])]
rna_cd8_adata = rna_cd8_adata[rna_cd8_adata.obs['LS_L4'].isin(['CD8_naive_SOX4-','CD8_naive_SOX4+'])]

In [18]:
cd4expressedgenes = (np.max(rna_cd4_adata.raw.X, 0).todense() > 0)

cd4expressedgenes = list(np.array(cd4expressedgenes)[0])

cd4expressedgenes = rna_cd4_adata.raw.var.index[cd4expressedgenes]

In [19]:
cd8expressedgenes = (np.max(rna_cd8_adata.raw.X, 0).todense() > 0)

cd8expressedgenes = list(np.array(cd8expressedgenes)[0])

cd8expressedgenes = rna_cd8_adata.raw.var.index[cd8expressedgenes]

In [20]:
expressedtfs = dict()

for cur in cd4tfdict.keys():
    varsplit = cur.split("(")
    complexsplit = varsplit[0].split("::")
    expressedtfs[cur] = complexsplit

In [21]:
ncd4tfs = []

for cur in expressedtfs.keys():
    curtfgenes = expressedtfs[cur]
    for curtfgene in curtfgenes:
        if curtfgene in cd4expressedgenes:
            ncd4tfs.append(cur)
            break
            
ncd8tfs = []
for cur in expressedtfs.keys():
    curtfgenes = expressedtfs[cur]
    for curtfgene in curtfgenes:
        if curtfgene in cd8expressedgenes:
            ncd8tfs.append(cur)
            break        

In [23]:
cd4_paired_summary = getPairedDeviationsSummary(cd4, "CD4_naive_SOX4-", "CD4_naive_SOX4+", cd4tfdict, ncd4tfs)

/Users/athib/Library/Python/3.13/lib/python/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/athib/Library/Python/3.13/lib/python/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [24]:
from statsmodels.stats import multitest

In [25]:
cd4_paired_summary['padjust'] = multitest.fdrcorrection(cd4_paired_summary['Wilcoxon'])[1]

In [26]:
cd4_paired_summary.sort_values('Wilcoxon').to_csv("NaiveCD4_chromVAR.csv")

In [27]:
cd8_paired_summary = getPairedDeviationsSummary(cd8, "CD8_naive_SOX4-", "CD8_naive_SOX4+", cd8tfdict, ncd8tfs)

In [28]:
cd8_paired_summary['padjust'] = multitest.fdrcorrection(cd8_paired_summary['Wilcoxon'])[1]

In [29]:
cd8_paired_summary.sort_values('Wilcoxon').to_csv("NaiveCD8_chromVAR.csv")